# DeltaNet v2: Performance Benchmark on Google Colab (Tesla T4)
### Comparing DeltaNet v2 (SM75 CuTe) against NVIDIA GatedDeltaNet-2 and FLA

This notebook downloads and executes the benchmark suite from `/content/Deltanet/tests/` to evaluate:
- **Runtime Performance**: Latency (ms) and Throughput (tokens/sec)
- **Peak Memory Consumption**: VRAM allocated & reserved (MB)

**Baselines Compared:**
1. **DeltaNet v2 (Ours)**: Fused CUDA kernels targeting NVIDIA Turing SM75 (Tesla T4, RTX 20-series) with independent $k, v$ gating.
2. **FLA GatedDeltaNet-2**: NVIDIA's official Gated DeltaNet-2 paper architecture implemented via Flash Linear Attention (`fla.layers.GatedDeltaNet2`).
3. **FLA GatedDeltaNet (v1)**: Standard Gated DeltaNet linear attention layer (`fla.layers.GatedDeltaNet`).
4. **PyTorch Reference**: Unfused PyTorch eager baseline.

## 1. Verify GPU Hardware
Confirm that Google Colab is running with an **NVIDIA Tesla T4** GPU (SM75, 16 GB VRAM).

In [ ]:
!nvidia-smi

## 2. Clone Repository & Build DeltaNet v2
Clones `https://github.com/CBG6682/Varlen-Gated-DeltaNet.git` into `/content/Deltanet` and compiles the SM75 fused CUDA extensions.

In [ ]:
# Clone into /content/Deltanet
!rm -rf /content/Deltanet
!git clone https://github.com/CBG6682/Varlen-Gated-DeltaNet.git /content/Deltanet
%cd /content/Deltanet

# Install build dependencies and compile DeltaNet v2
!pip install -q ninja setuptools wheel einops
!pip install -e . --no-build-isolation

## 3. Install Baseline Libraries
Installs **Flash Linear Attention** (`fla`) and **NVIDIA GatedDeltaNet-2** (`NVlabs/GatedDeltaNet-2`).

In [ ]:
# Install Flash Linear Attention (FLA)
!pip install -q flash-linear-attention

# Install NVlabs GatedDeltaNet-2
!pip install -q git+https://github.com/nvlabs/gateddeltanet-2.git --no-deps || true

## 4. Numerical Correctness Sanity Test
Verifies numerical consistency between DeltaNet v2 CUDA kernels and the pure PyTorch reference.

In [ ]:
!python /content/Deltanet/tests/test_correctness.py

## 5. Benchmark: Autoregressive Inference (L = 1)
Measures single-step decode latency (ms/step), generation throughput (tokens/sec), and peak VRAM across batch sizes $B \in [1, 16, 64, 256, 1024, 2048]$.

In [ ]:
!python /content/Deltanet/tests/bench_inference.py --dim 768 --q-heads 8 --kv-heads 8 --batches 1,16,64,256,1024,2048

## 6. Benchmark: Training Pass (Forward + Backward)
Measures step time (ms), training throughput (tokens/sec), and peak VRAM across sequence lengths $L \in [512, 1024, 2048, 4096]$.

In [ ]:
!python /content/Deltanet/tests/bench_training.py --dim 768 --q-heads 8 --kv-heads 8 --batch-size 4 --seq-lens 512,1024,2048,4096

## 7. Run Full Benchmark Suite
Runs all benchmarks and prints consolidated comparison tables.

In [ ]:
!python /content/Deltanet/tests/bench_all.py